In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

import matplotlib.pyplot as plt

: 

In [ ]:
def dj_oracle(n, balanced=False):

    oracle = QuantumCircuit(n + 1)

    if balanced:
        for qubit in range(n):
            oracle.cx(qubit, n)

    return oracle

In [ ]:
def deutsch_jozsa_circuit(n, balanced=True):

    qc = QuantumCircuit(n + 1, n)

    qc.x(n)

    qc.h(range(n + 1))

    oracle = dj_oracle(n, balanced)
    qc.compose(oracle, inplace=True)

    qc.h(range(n))

    qc.measure(range(n), range(n))

    return qc

In [ ]:
n = 3

qc = deutsch_jozsa_circuit(n, balanced=True)

qc.draw("mpl")

In [ ]:
simulator = AerSimulator()

job = simulator.run(qc, shots=1024)

result = job.result()

counts_local = result.get_counts()

print(counts_local)

In [ ]:
from IPython.display import display

fig = plot_histogram(counts_local)
display(fig)

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token="IBM_API_KEY"
)

print("Connected successfully!")

In [ ]:
backend = service.least_busy(
    operational=True,
    simulator=False
)

print(backend)

In [ ]:
from qiskit import transpile

transpiled_qc = transpile(
    qc,
    backend=backend,
    optimization_level=1
)

transpiled_qc.draw("mpl")

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler

sampler = Sampler(backend)

job = sampler.run([transpiled_qc], shots=1024)

print("Job submitted!")

In [ ]:
result = job.result()

counts_ibm = result[0].data.c.get_counts()

print(counts_ibm)